# 🔗 Phase 3 — Multimodal Late Fusion (experimental)

> **This phase is experimental, and reports no accuracy gain (ADR-002).**
> No public dataset pairs a leaf photograph with the variety, batch and
> planting date of that same plant, so the metadata used to train the
> fusion model is invented — every row of `labels_with_metadata.csv` is
> flagged `synthetic=1`. A model trained on invented metadata cannot show
> that metadata helps; it can only show whether the architecture runs.
> The ablation at the end of this notebook is what decides the question,
> and on synthetic data it comes out as noise, which is the correct answer.

The architecture combines:
- **Visual branch:** EfficientNet-B3 feature extractor producing a 256-d embedding.
- **Metadata branch:** Dense(32) + BatchNorm over the 17-d seed-label vector.
- **Fusion head:** concatenate (288-d) → Dense(128) → Dropout → softmax(4).


In [1]:
# Setup & Imports
import sys
import os
from pathlib import Path

PROJECT_ROOT = Path("..").resolve() if Path(".").resolve().name == "notebooks" else Path(".").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)

import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt
import tensorflow as tf

from src.phase1_cnn.data_pipeline import CLASS_NAMES, IMG_SIZE
from src.phase2_ocr.encoder import encode, METADATA_DIM
from src.phase3_fusion.fusion_model import build_fusion_model

print("TensorFlow:", tf.__version__)
print("Metadata Dimension:", METADATA_DIM)


TensorFlow: 2.16.2
Metadata Dimension: 17


### 1. Build and Inspect the Multimodal Fusion Model

In [2]:
# Build fusion model with Phase 1 CNN weights
CNN_WEIGHTS = "models/checkpoints/phase1_stage2_best.keras"
fusion_model = build_fusion_model(cnn_weights_path=CNN_WEIGHTS, freeze_cnn=True)

print("=" * 65)
print("MULTIMODAL FUSION ARCHITECTURE SUMMARY:")
print("=" * 65)
for layer in fusion_model.layers:
    trainable_str = "Trainable" if layer.trainable else "Frozen"
    print(f"{layer.name:<24} | Output: {str(layer.output.shape):<22} | {trainable_str}")
print("=" * 65)


Loaded CNN model from models/checkpoints/phase1_stage2_best.keras
MULTIMODAL FUSION ARCHITECTURE SUMMARY:
metadata_input           | Output: (None, 17)             | Trainable
image_input              | Output: (None, 300, 300, 3)    | Trainable
meta_dense               | Output: (None, 32)             | Trainable
feature_extractor        | Output: (None, 256)            | Trainable
meta_bn                  | Output: (None, 32)             | Trainable
fusion_concat            | Output: (None, 288)            | Trainable
fusion_dense_128         | Output: (None, 128)            | Trainable
fusion_dropout           | Output: (None, 128)            | Trainable
predictions              | Output: (None, 4)              | Trainable


### 2. Dual-Input Inference Demonstration

In [3]:
# Load a real leaf sample
test_img_path = "data/raw/plantvillage/data/Common_Rust/Corn_Common_Rust (921).JPG"
img = Image.open(test_img_path).resize(IMG_SIZE)
img_arr = np.expand_dims(np.array(img, dtype=np.float32), 0)

# Simulate two distinct agricultural field contexts for the exact same leaf:
# Context A: Resistant variety planted in dry conditions
meta_a = encode({
    "crop_variety": "SAMMAZ 15",
    "batch_number": "LOT-2024-001",
    "planting_date": "2024-06-01"
})

# Context B: Highly susceptible variety planted in peak humid season
meta_b = encode({
    "crop_variety": "OBA SUPER 2",
    "batch_number": "LOT-2023-088",
    "planting_date": "2023-11-01"
})

meta_batch_a = np.expand_dims(meta_a, 0)
meta_batch_b = np.expand_dims(meta_b, 0)

pred_a = fusion_model.predict([img_arr, meta_batch_a], verbose=0)[0]
pred_b = fusion_model.predict([img_arr, meta_batch_b], verbose=0)[0]

# Plot Comparison
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))

ax1.barh(CLASS_NAMES, pred_a, color="#3498db")
ax1.set_title("Prediction with Context A (SAMMAZ 15)", fontsize=11)
ax1.set_xlim(0, 1.05)
for i, v in enumerate(pred_a):
    ax1.text(v + 0.02, i, f"{v:.1%}", va="center")

ax2.barh(CLASS_NAMES, pred_b, color="#e67e22")
ax2.set_title("Prediction with Context B (OBA SUPER 2)", fontsize=11)
ax2.set_xlim(0, 1.05)
for i, v in enumerate(pred_b):
    ax2.text(v + 0.02, i, f"{v:.1%}", va="center")

plt.suptitle("Multimodal Fusion Inference on Identical Leaf", fontsize=13, y=1.02)
plt.tight_layout()
plt.show()


<Figure size 1400x400 with 2 Axes>

### 3. Ablation — does the metadata carry anything?


In [4]:
# Recorded by: python -m src.phase3_fusion.ablation --epochs 3
# Three arms on one split: zeroed metadata, the real vectors, and the same
# vectors permuted across images (marginals kept, pairing destroyed).
import json

with open("models/exports/metrics.json") as f:
    metrics = json.load(f)

ablation = metrics.get("fusion_ablation")
if not ablation:
    print("No ablation recorded yet — run:")
    print("  python -m src.phase3_fusion.ablation --epochs 3")
else:
    for arm, result in ablation["arms"].items():
        print(f"{arm:16s} {result['accuracy'] * 100:6.2f}%  (loss {result['loss']:.4f}, n={result['n']})")
    print(f"\nmetadata synthetic: {ablation['metadata_is_synthetic']}")
    print(ablation["verdict"]["summary"])
    print("\nFusion must beat the shuffled arm, not merely the zeroed one:")
    print("beating the zeroed arm alone would show that the extra parameters")
    print("helped, not that the metadata said anything.")


cnn_only          93.63%  (loss 0.1640, n=628)
fusion            93.79%  (loss 0.1638, n=628)
fusion_shuffled   93.63%  (loss 0.1690, n=628)

metadata synthetic: True
All three arms are within noise (fusion vs CNN +0.002, vs shuffled +0.002, band 0.010).

Fusion must beat the shuffled arm, not merely the zeroed one:
beating the zeroed arm alone would show that the extra parameters
helped, not that the metadata said anything.
